In [17]:
import pandas as pd
import numpy as np
from sklearn import linear_model #линейные моделиё
from sklearn import ensemble #ансамбли
from sklearn import metrics #метрики
from sklearn.model_selection import train_test_split #сплитование выборки
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
import optuna

In [2]:
#Загружаем данные и проверяем, что все работает
df = pd.read_csv('data/train.csv')
df.head()

,Activity,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,D1767,D1768,D1769,D1770,D1771,D1772,D1773,D1774,D1775,D1776
0,1,0.000000,0.497009,0.10,0.0,0.132956,0.678031,0.273166,0.585445,0.743663,...,0,0,0,0,0,0,0,0,0,0
1,1,0.366667,0.606291,0.05,0.0,0.111209,0.803455,0.106105,0.411754,0.836582,...,1,1,1,1,0,1,0,0,1,0
2,1,0.033300,0.480124,0.00,0.0,0.209791,0.610350,0.356453,0.517720,0.679051,...,0,0,0,0,0,0,0,0,0,0
3,1,0.000000,0.538825,0.00,0.5,0.196344,0.724230,0.235606,0.288764,0.805110,...,0,0,0,0,0,0,0,0,0,0
4,0,0.100000,0.517794,0.00,0.0,0.494734,0.781422,0.154361,0.303809,0.812646,...,0,0,0,0,0,0,0,0,0,0


In [3]:
X = df.drop(['Activity'], axis=1)
y = df['Activity']

In [4]:
#Делаем стратифицированное разбиение выборок в соотношении 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42, test_size=0.2)

In [18]:
random_state = 42

### <center> Линейная регрессия <center>

In [11]:
#Инициализируем модель лог. регрессии
log_reg = linear_model.LogisticRegression()

#Обучаем модель
log_reg.fit(X_train, y_train)
#Делаем предсказание для тренировочной выборки
y_train_pred = log_reg.predict(X_train)
print("F1 на тренировочном наборе: {:.2f}".format(metrics.f1_score(y_train, y_train_pred)))
#Делаем предсказание для тестовой выборки
y_test_pred = log_reg.predict(X_test)
print("F1 на тестовом наборе: {:.2f}".format(metrics.f1_score(y_test, y_test_pred)))

F1 на тренировочном наборе: 0.89
F1 на тестовом наборе: 0.78


c:\Users\Кирилл\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [16]:
param_grid = {'penalty': ['l2', None] ,
              'solver': ['lbfgs', 'sag'],
               'C': list(np.linspace(0.01, 1, 10, dtype=float))}

grid_search = GridSearchCV(
    estimator=linear_model.LogisticRegression(
        random_state=random_state,
        max_iter=1000
    ),
    param_grid=param_grid,
    cv=5,
    n_jobs=-1
)

%time grid_search.fit(X_train, y_train)
y_test_pred = grid_search.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(grid_search.best_params_))

CPU times: total: 32.8 s
Wall time: 1min 56s
f1_score на тестовом наборе: 0.78
Наилучшие значения гиперпараметров: {'C': np.float64(0.01), 'penalty': 'l2', 'solver': 'lbfgs'}


c:\Users\Кирилл\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


Показатели метрики улучшить не удалось, лучшие параметры по GridSearch: 

penalty: l2 

solver: lbfgs

C: 0.01

In [15]:
param_distributions = {'penalty': ['l2', None] ,
              'solver': ['lbfgs', 'sag'],
               'C': list(np.linspace(0.01, 1, 10, dtype=float))}

random_search = RandomizedSearchCV(
    estimator= linear_model.LogisticRegression(
        random_state=random_state,
        max_iter=1000
    ),
    param_distributions=param_distributions,
    cv=5,
    n_iter=10,
    n_jobs=-1
)

%time random_search.fit(X_train, y_train)
y_test_pred = random_search.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(random_search.best_params_))

CPU times: total: 8.86 s
Wall time: 31.7 s
f1_score на тестовом наборе: 0.79
Наилучшие значения гиперпараметров: {'solver': 'lbfgs', 'penalty': 'l2', 'C': np.float64(0.12)}


c:\Users\Кирилл\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


С рандомным поиском удалось слегка улучшить показатель метрики F1, лучшие параметры: solver: lbfgs, penalty: l2, C: 0.12

In [21]:
def optuna_log_reg(trial):
    # задаем пространства поиска гиперпараметров
    C = trial.suggest_float("C", 0.001, 10, log=True)
    solver = trial.suggest_categorical("solver", ["lbfgs", "saga"])

    # penalty зависит от solver
    if solver == "lbfgs":
        penalty = "l2"
        l1_ratio = None
    else:  # saga
        penalty = trial.suggest_categorical("penalty", ["l1", "l2"])
        l1_ratio = None

    model = linear_model.LogisticRegression(
        C=C,
        solver=solver,
        penalty=penalty,
        random_state=random_state,
        max_iter=1000
    )

    model.fit(X_train, y_train)
    score = metrics.f1_score(y_train, model.predict(X_train))

    return score

In [22]:
study = optuna.create_study(study_name='LogisticRegression', direction='maximize')
study.optimize(optuna_log_reg, n_trials=20)

[I 2026-02-24 16:17:01,132] A new study created in memory with name: LogisticRegression
c:\Users\Кирилл\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
[I 2026-02-24 16:17:01,366] Trial 0 finished with value: 0.8619548872180451 and parameters: {'C': 0.15785401777566258, 'solver': 'lbfgs'}. Best is trial 0 with value: 0.8619548872180451.
c:\Users\Кирилл\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. 

In [23]:
print("Best F1:", study.best_value)
print("Best params:", study.best_params)

Best F1: 0.9303467321264192
Best params: {'C': 9.792717938433041, 'solver': 'lbfgs'}


Значение метрики значительно увеличилось с помощью optuna

Параметры:

С: 9.792717938433041,

solver: lbfgs

### <center> Случайный лес <center>

In [24]:
rf = ensemble.RandomForestClassifier()

rf.fit(X_train, y_train)

y_train_pred = rf.predict(X_train)
print('Train: {:.2f}'.format(metrics.f1_score(y_train, y_train_pred)))
y_test_pred = rf.predict(X_test)
print('Test: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))

Train: 1.00
Test: 0.80


С дефолтными параметрами метрика составила 0.80, что уже очень даже хорошо, сейчас посмотрим получится ли ее улучшить

In [26]:
param_distributions_forest = {'n_estimators': list(range(100, 200, 30)),
              'min_samples_leaf': [5, 7],
              'max_depth': list(np.linspace(10, 15, 20, 25, dtype=int))
              }

grid_search_rf = GridSearchCV(
    estimator=ensemble.RandomForestClassifier(random_state=random_state),
    param_grid=param_distributions_forest,
    n_jobs=-1,
    cv=5
)

%time grid_search_rf.fit(X_train, y_train)
y_test_pred = grid_search_rf.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(grid_search_rf.best_params_))

CPU times: total: 1min 38s
Wall time: 1min 48s
f1_score на тестовом наборе: 0.80
Наилучшие значения гиперпараметров: {'max_depth': np.int64(15), 'min_samples_leaf': 5, 'n_estimators': 190}


Показатель метрики улучшить не удалось, лучшие параметры:

max_depth: 15,

min_samples_leaf: 5,

n_estimators: 190

In [27]:
random_search_rf = RandomizedSearchCV(
    estimator=ensemble.RandomForestClassifier(random_state=random_state),
    param_distributions=param_distributions_forest,
    cv=5,
    n_iter=10,
    n_jobs=-1
)

%time random_search_rf.fit(X_train, y_train)
y_test_pred = random_search_rf.predict(X_test)
print('f1_score на тестовом наборе: {:.2f}'.format(metrics.f1_score(y_test, y_test_pred)))
print("Наилучшие значения гиперпараметров: {}".format(random_search_rf.best_params_))

CPU times: total: 6.78 s
Wall time: 8.16 s
f1_score на тестовом наборе: 0.80
Наилучшие значения гиперпараметров: {'n_estimators': 190, 'min_samples_leaf': 5, 'max_depth': np.int64(11)}


Метрику так же не удалось улучшить, параметры остались те же, но времени затрачено в разы меньше

In [32]:
def optuna_rf(trial):
    # задаем пространства поиска гиперпараметров
    n_estimators = trial.suggest_int('n_estimators', 100, 200, 1)
    max_depth = trial.suggest_int('max_depth', 10, 30, 1)
    min_sample_leaf = trial.suggest_int('min_samples_leaf', 2, 10, 1)

    #Создаем модель
    model = ensemble.RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_leaf=min_sample_leaf,
        random_state=random_state
    )

    model.fit(X_train, y_train)
    score = metrics.f1_score(y_test, model.predict(X_test))

    return score

In [33]:
study = optuna.create_study(study_name="RandomForestClassifier", direction="maximize")
study.optimize(optuna_rf, n_trials=20)

[I 2026-02-24 16:42:53,352] A new study created in memory with name: RandomForestClassifier
C:\Users\Кирилл\AppData\Local\Temp\ipykernel_21148\2131932936.py:3: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with the corresponding keyword arguments in v5.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v3.5.0 for details.
  n_estimators = trial.suggest_int('n_estimators', 100, 200, 1)
C:\Users\Кирилл\AppData\Local\Temp\ipykernel_21148\2131932936.py:4: FutureWarning: suggest_int() got {'step'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'name', 'low', 'high', 'step', 'log'] in suggest_int() have been deprecated since v3.5.0. They will be replaced with th

In [35]:
print("Best F1:", round(study.best_value,2))
print("Best params:", study.best_params)

Best F1: 0.81
Best params: {'n_estimators': 158, 'max_depth': 28, 'min_samples_leaf': 4}


Значение метрики увеличилось совсем немного, оптимальные параметры:

n_estimators: 158

max_depth: 28

min_samples_leaf: 4